In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import resample_poly
import os
from scipy.signal import butter, filtfilt
import joblib
from sklearn.preprocessing import StandardScaler

In [2]:
sf_dict = {"fog_star": 60.0, "omnia_park": 90.0, "pd_phone": 200.0, "wearpd": 100.0}
datasets = ["fog_star", "omnia_park", "pd_phone", "wearpd"]
target_hz = 128.0

In [37]:
def enforce_nan_policy(group, sf, max_interp_gap_sec=0.2, sensor_cols=None):
    if sensor_cols is None:
        sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']

    group = group.sort_values('timestamp').copy()
    max_interp_gap = max(1, int(max_interp_gap_sec * sf))

    for col in sensor_cols:
        isna = group[col].isna().to_numpy()

        if isna.any():
            # Find contiguous NaN blocks and reject the whole session if one is too long.
            edges = np.diff(np.r_[False, isna, False].astype(int))
            starts = np.where(edges == 1)[0]
            ends = np.where(edges == -1)[0]
            max_gap = int((ends - starts).max())

            if max_gap > max_interp_gap:
                return None

            group[col] = group[col].interpolate(method='linear', limit_direction='both')

        # Safety check: if interpolation could not fill everything, reject session.
        if group[col].isna().any():
            return None

    return group


def trim_outliers(group, sf, z_threshold=2.5, trim_perc=0.15, window_size_sec=3.0):
    group = group.sort_values('timestamp').copy()

    total_duration = group['timestamp'].max() - group['timestamp'].min()
    n_trim = int(trim_perc * total_duration * sf)

    if len(group) < (2 * n_trim + sf):
        return None
    group = group.iloc[n_trim:-n_trim].copy()

    sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    window_size = max(3, int(window_size_sec * sf))

    for col in sensor_cols:
        rolling = group[col].rolling(window=window_size, center=True, min_periods=1)
        z_score = np.abs((group[col] - rolling.mean()) / (rolling.std() + 1e-6))
        group.loc[z_score > z_threshold, col] = np.nan

    # Apply strict NaN policy after outlier removal.
    group = enforce_nan_policy(group, sf, max_interp_gap_sec=0.2, sensor_cols=sensor_cols)
    if group is None:
        return None

    group['timestamp'] = (group['timestamp'] - group['timestamp'].min()).round(4)
    return group

In [38]:
def lowpass_filter(group, sf, cutoff=5.0, order=4):
    nyquist = 0.5 * sf
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)

    sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    
    for col in sensor_cols:
        mean_val = group[col].mean()
        signal_centered = group[col].values - mean_val
        filtered_centered = filtfilt(b, a, signal_centered)
        group[col] = filtered_centered + mean_val
        
    return group

In [39]:
def resample(group, original_sf, target_sf=128.0):
    sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    up, down = int(target_sf), int(original_sf)
    
    pad_samples = int(original_sf) 
    n_orig = len(group)
    n_target = int(n_orig * target_sf / original_sf)

    resampled_data = {
        'timestamp': np.linspace(0, (n_orig-1)/original_sf, n_target),
        'subjectID': group['subjectID'].iloc[0],
        'sessionID': group['sessionID'].iloc[0],
        'taskID': group['taskID'].iloc[0]
    }

    for col in sensor_cols:
        padded = np.pad(group[col].values, pad_width=pad_samples, mode='reflect')
        
        resampled_padded = resample_poly(padded, up, down)
        
        pad_target = int(pad_samples * target_sf / original_sf)
        resampled_data[col] = resampled_padded[pad_target : pad_target + n_target]
    
    return pd.DataFrame(resampled_data)

In [43]:
os.makedirs("data/preprocessed_data", exist_ok=True)

sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']

for dataset in datasets:
    path = f"data/cleaned_data/{dataset}_sensor.csv"
    if not os.path.exists(path):
        continue

    print(f"\n--- Processing {dataset} ---")
    df = pd.read_csv(path)
    sf = sf_dict[dataset]

    df_filtered = df[df.taskID.isin([0, 1, 2])].copy()
    processed_sessions = []
    total_count, discarded_count = 0, 0

    grouped = df_filtered.groupby(['subjectID', 'sessionID', 'taskID'])

    for _, session_group in grouped:
        total_count += 1

        if session_group['taskID'].iloc[0] in [0, 1]:
            # Task 0,1: trim outliers + strict NaN policy + lowpass + resample
            temp_group = trim_outliers(session_group, sf)
            if temp_group is None:
                discarded_count += 1
                continue
        else:
            # Task 2: no outlier trim, but same strict NaN policy
            temp_group = session_group.sort_values('timestamp').iloc[int(sf):-int(sf)].copy()
            if len(temp_group) <= sf:
                discarded_count += 1
                continue

            temp_group = enforce_nan_policy(temp_group, sf, max_interp_gap_sec=0.2, sensor_cols=sensor_cols)
            if temp_group is None:
                discarded_count += 1
                continue

            temp_group['timestamp'] = (temp_group['timestamp'] - temp_group['timestamp'].min()).round(4)

        temp_group = lowpass_filter(temp_group, sf, cutoff=5.0)
        temp_group = resample(temp_group, sf, target_sf=target_hz)

        # Final hard check: never keep sessions with NaNs in sensor features.
        if temp_group[sensor_cols].isna().any().any():
            discarded_count += 1
            continue

        processed_sessions.append(temp_group)

    if processed_sessions:
        out_df = pd.concat(processed_sessions, ignore_index=True)

        # Keep only fully usable processed sessions (session-level NaN-free check).
        valid_mask = (
            out_df.groupby(['subjectID', 'sessionID', 'taskID'])[sensor_cols]
            .transform(lambda x: ~x.isna().any())
            .all(axis=1)
        )
        out_df = out_df.loc[valid_mask].copy()
        out_df = out_df.dropna(subset=sensor_cols)

        out_df.to_csv(f"data/preprocessed_data/{dataset}_sensor.csv", index=False)
        print(f"Saved: {total_count - discarded_count}/{total_count} sessions.")
        print(f"Final usable rows: {len(out_df)}")
    else:
        print(f"No valid sessions for {dataset} after quality checks.")


--- Processing fog_star ---
timestamp    0
subjectID    0
sessionID    0
taskID       0
acc_x        0
acc_y        0
acc_z        0
gyro_x       0
gyro_y       0
gyro_z       0
dtype: int64
Saved: 16/16 sessions.
Final usable rows: 67409

--- Processing omnia_park ---
timestamp    0
subjectID    0
sessionID    0
taskID       0
acc_x        0
acc_y        0
acc_z        0
gyro_x       0
gyro_y       0
gyro_z       0
dtype: int64
Saved: 79/121 sessions.
Final usable rows: 822388

--- Processing pd_phone ---
timestamp    0
subjectID    0
sessionID    0
taskID       0
acc_x        0
acc_y        0
acc_z        0
gyro_x       0
gyro_y       0
gyro_z       0
dtype: int64
Saved: 42/42 sessions.
Final usable rows: 127708

--- Processing wearpd ---
timestamp    0
subjectID    0
sessionID    0
taskID       0
acc_x        0
acc_y        0
acc_z        0
gyro_x       0
gyro_y       0
gyro_z       0
dtype: int64
Saved: 292/293 sessions.
Final usable rows: 701563


In [ ]:
for dataset in datasets:
    path_pre = f"data/preprocessed_data/{dataset}_sensor.csv"
    path_orig = f"data/cleaned_data/{dataset}_sensor.csv"
    
    if not (os.path.exists(path_pre) and os.path.exists(path_orig)):
        continue
        
    df_pre = pd.read_csv(path_pre)
    df_orig = pd.read_csv(path_orig)

    
    # Raggruppiamo per sessione
    grouped = df_pre.groupby(["subjectID", "sessionID", "taskID"])
    
    for (sub_id, sess_id, task_id), group_pre in grouped:
        # Recupero sessione originale corrispondente
        group_orig = df_orig[
            (df_orig["subjectID"] == sub_id) & 
            (df_orig["sessionID"] == sess_id) & 
            (df_orig["taskID"] == task_id)
        ].sort_values("timestamp")

        if group_orig.empty:
            continue

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
        task_label = "Eyes Open" if task_id == 0 else "Eyes Closed"

        for col in ['acc_x', 'acc_y', 'acc_z']:
            ax1.plot(group_orig["timestamp"], group_orig[col], label=col, alpha=0.5)
        
        ax1.set_title(f"Original: {dataset}\nSub {sub_id}, Task {task_id} ({task_label})")
        ax1.set_xlabel("Time (s)")
        ax1.set_ylabel("Accel (g)")
        ax1.legend(loc='upper right')
        ax1.grid(True, alpha=0.3)
        colors = plt.rcParams['axes.prop_cycle'].by_key()['color'][:3]
        
        for i, col in enumerate(['acc_x', 'acc_y', 'acc_z']):
            ax2.plot(group_pre["timestamp"], group_pre[col], color=colors[i], label=f"{col} Cleaned")
        ax2.set_title(f"Preprocessed")
        ax2.set_xlabel("Time (s)")
        ax2.grid(True, alpha=0.3)
        ax2.legend(loc='upper right')

        plt.tight_layout()
        plt.show()

In [3]:
def create_windows(df, window_size=256, overlap=0.5):
    X = []
    meta = []
    step = int(window_size * (1 - overlap))
    sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    
    # Raggruppiamo per sessione reale per non mischiare i dati
    grouped = df.groupby(['subjectID', 'sessionID', 'taskID'])
    
    for (sub_id, sess_id, task_id), group in grouped:
        data = group[sensor_cols].values
        if len(data) < window_size:
            continue
            
        for i in range(0, len(data) - window_size + 1, step):
            X.append(data[i : i + window_size])
            meta.append({
                'subjectID': sub_id,
                'sessionID': sess_id,
                'taskID': task_id,
                'dataset': group['dataset'].iloc[0]
            })
            
    return np.array(X), pd.DataFrame(meta)

In [4]:
all_dfs = []
sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']

for dataset in datasets:
    path = f"data/preprocessed_data/{dataset}_sensor.csv"
    if os.path.exists(path):
        temp_df = pd.read_csv(path)
        temp_df['dataset'] = dataset

        # Keep only fully usable sessions (no NaN in any sensor channel).
        valid_mask = (
            temp_df.groupby(['subjectID', 'sessionID', 'taskID'])[sensor_cols]
            .transform(lambda x: ~x.isna().any())
            .all(axis=1)
        )
        temp_df = temp_df.loc[valid_mask].copy().dropna(subset=sensor_cols)

        all_dfs.append(temp_df)

full_df = pd.concat(all_dfs, ignore_index=True)
print(full_df.isna().sum())  # Deve essere tutto 0

X_raw, metadata = create_windows(full_df, window_size=256, overlap=0.5)
n_windows, w_size, n_channels = X_raw.shape
X_flat = X_raw.reshape(-1, n_channels)

scaler = StandardScaler()
X_scaled_flat = scaler.fit_transform(X_flat)
X_final = X_scaled_flat.reshape(n_windows, w_size, n_channels)

axes_dir = "data/windowed_data/axes"
os.makedirs(axes_dir, exist_ok=True)

# save each axis separately
for i, col in enumerate(sensor_cols):
    axis_data = X_final[:, :, i].reshape(n_windows, 1, w_size)
    np.save(f"{axes_dir}/{col}.npy", axis_data.astype(np.float32))

# Salva comunque il metadato e lo scaler
metadata.to_csv("data/windowed_data/metadata.csv", index=False)
joblib.dump(scaler, "data/windowed_data/scaler.joblib")

timestamp    0
subjectID    0
sessionID    0
taskID       0
acc_x        0
acc_y        0
acc_z        0
gyro_x       0
gyro_y       0
gyro_z       0
dataset      0
dtype: int64


['data/windowed_data/scaler.joblib']